In [49]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer, Normalizer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss, log_loss

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
print('CWD:', Path.cwd())

ROOT = Path.cwd().resolve()
OUT_DIR = ROOT / 'out_data'
if not OUT_DIR.exists():
    hits = list(ROOT.rglob('out_data'))
    if hits:
        OUT_DIR = hits[0]

print('OUT_DIR:', OUT_DIR)

X_FILE = OUT_DIR / 'binary_vosa_XPcoeff_110d_L2.csv'
if not X_FILE.exists():
    raise FileNotFoundError(
        f'Missing {X_FILE}\n'
        'Expected the artifact created in the data-construction notebook.\n'
        'Make sure out_data/binary_vosa_XPcoeff_110d_L2.csv exists.'
    )

df_all = pd.read_csv(X_FILE)
feat_cols = [c for c in df_all.columns if re.fullmatch(r'c\d{3}', c)]

X = df_all[feat_cols].to_numpy(dtype=np.float32)
y = df_all['y'].to_numpy(dtype=np.int64)
source_ids = df_all['source_id'].to_numpy(dtype=np.int64)

print('Loaded:', X_FILE.name)
print('Shape :', df_all.shape)
print('Positives:', int(df_all['y'].sum()), '| Negatives:', int((1 - df_all['y']).sum()))
df_all.head()


In [51]:
# 70/15/15 (stratified)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp
)

print('Train:', X_train.shape, 'pos_rate:', y_train.mean())
print('Val  :', X_val.shape,   'pos_rate:', y_val.mean())
print('Test :', X_test.shape,  'pos_rate:', y_test.mean())


Train: (1970, 110) pos_rate: 0.19847715736040608
Val  : (422, 110) pos_rate: 0.1966824644549763
Test : (423, 110) pos_rate: 0.19858156028368795


In [52]:
# --- Metrics and helpers ---
def _conf_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    f1   = (2*prec*sens)/(prec+sens) if (prec+sens) else 0.0
    youden = sens + spec - 1.0
    return sens, spec, prec, f1, youden

def pick_threshold(y_true, p_pos, criterion='youden', grid_size=200):
    qs = np.linspace(0.0, 1.0, grid_size)
    thr_grid = np.unique(np.quantile(p_pos, qs))
    thr_grid = np.clip(thr_grid, 0.0, 1.0)

    best = None
    for thr in thr_grid:
        sens, spec, prec, f1, youden = _conf_metrics(y_true, (p_pos >= thr).astype(int))
        score = youden if criterion == 'youden' else f1
        row = (score, thr, sens, spec, prec)
        if best is None or row[0] > best[0]:
            best = row

    score, thr, sens, spec, prec = best
    return {'thr': float(thr), 'sens': sens, 'spec': spec, 'prec': prec, 'score': score}

def prob_metrics(y_true, p_pos):
    eps = 1e-15
    p = np.clip(p_pos, eps, 1 - eps)
    return {
        'ROC AUC': float(roc_auc_score(y_true, p)),
        'PR AUC': float(average_precision_score(y_true, p)),
        'Brier': float(brier_score_loss(y_true, p)),
        'Log loss': float(log_loss(y_true, p)),
    }

def style_results(df):
    metric_cols = ['Sensitivity', 'Specificity', 'Precision']
    fmt = {
        'Best k': '{:.0f}',
        'Best p': '{:.0f}',
        'Threshold': '{:.3f}',
        'Sensitivity': '{:.3f}',
        'Specificity': '{:.3f}',
        'Precision': '{:.3f}',
    }
    return (df.style
              .format(fmt, na_rep='')
              .background_gradient(cmap='viridis', subset=metric_cols)
           )

def style_prob_results(df):
    auc_cols  = ['ROC AUC', 'PR AUC']
    loss_cols = ['Brier', 'Log loss']
    fmt = {
        'Best k': '{:.0f}',
        'Best p': '{:.0f}',
        'ROC AUC': '{:.4f}',
        'PR AUC': '{:.4f}',
        'Brier': '{:.5f}',
        'Log loss': '{:.5f}',
    }
    return (df.style
              .format(fmt, na_rep='')
              .background_gradient(cmap='viridis', subset=auc_cols)
              .background_gradient(cmap='viridis_r', subset=loss_cols)
           )


In [53]:
# k values
K_GRID = [1,3,5,7,9,11,15,21,31,41,51,71,101]

# Transform options
TRANSFORMS = [
    None,
    'zscore',
    'robust',
    'power',
    'quantile',
    'l2',
    'pca30',
    'pca60',
    'pca90',
]

def make_transform(name):
    if name is None:
        return None
    if name == 'zscore':
        return StandardScaler()
    if name == 'robust':
        return RobustScaler()
    if name == 'power':
        return PowerTransformer(method='yeo-johnson')
    if name == 'quantile':
        return QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=RANDOM_STATE)
    if name == 'l2':
        return Normalizer(norm='l2')
    if name.startswith('pca'):
        n = int(name.replace('pca', ''))
        return Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=n, whiten=True, random_state=RANDOM_STATE))
        ])
    raise ValueError(f'Unknown transform: {name}')

# Build a compact, high-quality variant set (<=8)
VARIANTS = [
    {
        'name': 'kNN p=2 | distance | zscore',
        'weights': 'distance',
        'metric': 'minkowski',
        'p': 2,
        'transform': 'zscore',
    },
    {
        'name': 'kNN p=1 | distance | robust',
        'weights': 'distance',
        'metric': 'minkowski',
        'p': 1,
        'transform': 'robust',
    },
    {
        'name': 'kNN p=2 | distance | l2',
        'weights': 'distance',
        'metric': 'minkowski',
        'p': 2,
        'transform': 'l2',
    },
    {
        'name': 'kNN p=3 | distance | zscore',
        'weights': 'distance',
        'metric': 'minkowski',
        'p': 3,
        'transform': 'zscore',
    },
    {
        'name': 'kNN p=2 | uniform | pca60',
        'weights': 'uniform',
        'metric': 'minkowski',
        'p': 2,
        'transform': 'pca60',
    },
    {
        'name': 'kNN p=2 | distance | power',
        'weights': 'distance',
        'metric': 'minkowski',
        'p': 2,
        'transform': 'power',
    },
    {
        'name': 'kNN cosine | distance | l2',
        'weights': 'distance',
        'metric': 'cosine',
        'p': None,
        'transform': 'l2',
    },
    {
        'name': 'kNN cosine | distance | pca60',
        'weights': 'distance',
        'metric': 'cosine',
        'p': None,
        'transform': 'pca60',
    },
]

print('Total variants:', len(VARIANTS))

def fit_knn_variant(v, k):
    kwargs = {
        'n_neighbors': int(k),
        'weights': v['weights'],
        'metric': v['metric'],
    }
    if v['metric'] == 'minkowski':
        kwargs['p'] = int(v['p'])
    if v['metric'] == 'cosine':
        kwargs['algorithm'] = 'brute'
    knn = KNeighborsClassifier(**kwargs)

    transform = make_transform(v['transform'])
    if transform is None:
        model = knn
    else:
        model = Pipeline([('prep', transform), ('knn', knn)])

    model.fit(X_train, y_train)
    return model

def proba_positive(model, X):
    return model.predict_proba(X)[:,1]

def select_variant(v, criterion):
    best = None
    best_score = -1e9
    for k in K_GRID:
        model = fit_knn_variant(v, k)
        p_val = proba_positive(model, X_val)
        pick = pick_threshold(y_val, p_val, criterion=criterion)
        if pick['score'] > best_score:
            best_score = pick['score']
            best = {'model': model, 'bestk': int(k), 'thr': pick['thr']}
    return best

def eval_on_test(model, thr):
    p_test = proba_positive(model, X_test)
    sens, spec, prec, f1, youden = _conf_metrics(y_test, (p_test >= thr).astype(int))
    return {'Sensitivity': sens, 'Specificity': spec, 'Precision': prec}

def run_selection(criterion):
    rows = []
    sels = {}
    for v in VARIANTS:
        sel = select_variant(v, criterion=criterion)
        met = eval_on_test(sel['model'], sel['thr'])
        best_p = np.nan if v['p'] is None else int(v['p'])
        rows.append({
            'Variant': v['name'],
            'Best k': sel['bestk'],
            'Best p': best_p,
            'Threshold': sel['thr'],
            **met
        })
        sels[v['name']] = sel
    return pd.DataFrame(rows), sels

df_test_youden, sels_youden = run_selection('youden')
df_test_f1, sels_f1 = run_selection('f1')

# --- Benchmark comparison ---
BENCHMARK = {
    'Youden': {
        'Sensitivity': 0.878,
        'Specificity': 0.965,
        'Precision': 0.838,
    },
    'F1': {
        'Sensitivity': 0.818,
        'Specificity': 0.985,
        'Precision': 0.920,
    },
}

def delta_vs_benchmark(df, selection_label):
    metrics = ['Sensitivity', 'Specificity', 'Precision']
    base = BENCHMARK.get(selection_label, BENCHMARK.get('Youden', {m:0.0 for m in metrics}))
    deltas = df[['Variant'] + metrics].copy()
    for m in metrics:
        deltas[m] = deltas[m] - base[m]
    deltas.insert(0, 'Selection', selection_label)
    return deltas

# Show benchmark deltas FIRST (separate tables)
_delta_y = delta_vs_benchmark(df_test_youden, 'Youden')
_delta_f = delta_vs_benchmark(df_test_f1, 'F1')

max_abs_y = float(np.nanmax(np.abs(_delta_y[['Sensitivity','Specificity','Precision']].to_numpy())))
max_abs_f = float(np.nanmax(np.abs(_delta_f[['Sensitivity','Specificity','Precision']].to_numpy())))

print('\nΔ vs benchmark — Youden:')
display(
    _delta_y.style
    .format({'Sensitivity':'{:+.3f}', 'Specificity':'{:+.3f}', 'Precision':'{:+.3f}'})
    .background_gradient(cmap='BrBG', subset=['Sensitivity','Specificity','Precision'], vmin=-max_abs_y, vmax=max_abs_y)
)

print('\nΔ vs benchmark — F1:')
display(
    _delta_f.style
    .format({'Sensitivity':'{:+.3f}', 'Specificity':'{:+.3f}', 'Precision':'{:+.3f}'})
    .background_gradient(cmap='BrBG', subset=['Sensitivity','Specificity','Precision'], vmin=-max_abs_f, vmax=max_abs_f)
)

print('\nTEST — evaluated using VAL-picked (k, threshold) from Youden selection:')
display(style_results(df_test_youden.sort_values(['Specificity','Sensitivity','Precision'], ascending=False)))

print('\nTEST — evaluated using VAL-picked (k, threshold) from F1 selection:')
display(style_results(df_test_f1.sort_values(['Precision','Sensitivity','Specificity'], ascending=False)))

# Probability metrics table (Youden-picked models)
rows = []
for v in VARIANTS:
    sel = sels_youden[v['name']]
    pm = prob_metrics(y_test, proba_positive(sel['model'], X_test))
    best_p = np.nan if v['p'] is None else int(v['p'])
    rows.append({
        'Variant': v['name'],
        'Best k': sel['bestk'],
        'Best p': best_p,
        **pm
    })
df_prob_youden = pd.DataFrame(rows)

print('\nTEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) — Youden-picked models:')
display(style_prob_results(df_prob_youden.sort_values(['PR AUC','ROC AUC'], ascending=False)))


Total variants: 8


/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/Users/kris/Desktop/Astrophysics/.venv/lib/python3.9/


Δ vs benchmark — Youden:


,Selection,Variant,Sensitivity,Specificity,Precision
0,Youden,kNN p=2 | distance | zscore,-0.045,-0.174,-0.342
1,Youden,kNN p=1 | distance | robust,-0.176,-0.042,-0.144
2,Youden,kNN p=2 | distance | l2,-0.104,-0.012,-0.036
3,Youden,kNN p=3 | distance | zscore,-0.116,-0.104,-0.261
4,Youden,kNN p=2 | uniform | pca60,-0.152,-0.124,-0.308
5,Youden,kNN p=2 | distance | power,-0.092,-0.157,-0.334
6,Youden,kNN cosine | distance | l2,-0.104,-0.021,-0.064
7,Youden,kNN cosine | distance | pca60,-0.199,-0.039,-0.143



Δ vs benchmark — F1:


...

,Variant,Best k,Best p,Threshold,Sensitivity,Specificity,Precision
2,kNN p=2 | distance | l2,21,2,0.141,0.774,0.953,0.802
6,kNN cosine | distance | l2,21,,0.136,0.774,0.944,0.774
7,kNN cosine | distance | pca60,41,,0.223,0.679,0.926,0.695
1,kNN p=1 | distance | robust,21,1,0.135,0.702,0.923,0.694
3,kNN p=3 | distance | zscore,101,3,0.079,0.762,0.861,0.577
4,kNN p=2 | uniform | pca60,71,2,0.099,0.726,0.841,0.530
5,kNN p=2 | distance | power,71,2,0.059,0.786,0.808,0.504
0,kNN p=2 | distance | zscore,41,2,0.069,0.833,0.791,0.496



TEST — evaluated using VAL-picked (k, threshold) from F1 selection:


,Variant,Best k,Best p,Threshold,Sensitivity,Specificity,Precision
2,kNN p=2 | distance | l2,31,2,0.220,0.738,0.968,0.849
6,kNN cosine | distance | l2,31,,0.218,0.738,0.968,0.849
3,kNN p=3 | distance | zscore,11,3,0.452,0.571,0.968,0.814
0,kNN p=2 | distance | zscore,9,2,0.232,0.607,0.962,0.797
5,kNN p=2 | distance | power,11,2,0.274,0.631,0.959,0.791
7,kNN cosine | distance | pca60,21,,0.284,0.643,0.944,0.740
1,kNN p=1 | distance | robust,21,1,0.135,0.702,0.923,0.694
4,kNN p=2 | uniform | pca60,71,2,0.109,0.679,0.897,0.620



TEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) — Youden-picked models:


,Variant,Best k,Best p,ROC AUC,PR AUC,Brier,Log loss
6,kNN cosine | distance | l2,21,,0.8985,0.8120,0.06755,0.96583
2,kNN p=2 | distance | l2,21,2,0.8985,0.8112,0.06763,0.96621
5,kNN p=2 | distance | power,71,2,0.8838,0.7697,0.09284,0.46220
0,kNN p=2 | distance | zscore,41,2,0.8843,0.7665,0.09491,0.60992
7,kNN cosine | distance | pca60,41,,0.8735,0.7525,0.09240,0.31287
3,kNN p=3 | distance | zscore,101,3,0.8649,0.7423,0.10104,0.34677
1,kNN p=1 | distance | robust,21,1,0.8535,0.7341,0.09333,1.26495
4,kNN p=2 | uniform | pca60,71,2,0.8557,0.7071,0.11545,0.45282
